# Stiffness - aka "the thing that makes my simulation slow"

Stiffness is one of the main reasons why numerical simulations can be slow.
It is also a concept that is surprisingly difficult to define.
There is no single universally accepted mathematical definition, but several related mechanisms that can make a PDAE system stiff.

## What is it

In simple terms, a system is stiff when **the an explicit numerical solver must take very small time steps to maintain stability**, even though parts of the solution evolve much slower.
Smaller time steps mean more computations and, consequently, longer simulation times.
Implicit time integration is not limited by the time step size but require additional computations per time step.
Depending on the stability constraint on the explicit time step, implicit time integration can be faster, which is usually the case for chromatography simulations.

**Todo:** plot stability regions of two explicit methods, plot eigen values of a chromatography system and how the time step scales them into the stability region. Then, we say implicit time integration is completely stable and only accuracy dictates the time step.

<details>
<summary><b>Click to learn more about stability vs accuracy and explicit vs implicit time integration</b></summary>
Instability here refers to the accumulation of approximation errors of the numerical methods, such that the simulation "explodes", i.e. continuously increasing or decreasing values that blow-up the simulation.
This is different from accuracy constraints, which refer to an underresolved evolution of the simulation that may lead to completely wrong / inaccurate but stable results that are in a realistic order of magnitude.
todo - Example: minimalistic script creating plots - left plot with blown-up explicit solution, right plot with completely inaccurate implicit time integration solution, both useless but qualitatively different. refinement makes implicit continuously better with every refinement, but explicit has to overcome the stability constraint until it gets better.
Depending on the stability constraint on the explicit time step, implicit time integration can be faster, which is usually the case for chromatography simulations.
</details>

## Where does it come from

Mathematically, explicit time step restrictions (serving as an indicator of stiffness here) are related to the eigenvalues of the system, which describe how quickly different parts of the solution evolve.
The numerical time integration is forced to make smaller time steps (stability for explicit time integration, accuracy for implicit) to resolve the fast process and the whole simulation slows down.

1. One of the most common causes of stiffness is the presence of processes operating on very different time scales.
For example, adsorption or chemical reactions may occur almost instantaneously, while transport processes such as pore diffusion take much longer.

2. strong gradients: not the equations themselves but the current solution causes stiffness, e.g. strong concentration gradients.
Strong gradients can also make the nonlinear equations more difficult to solve.
Strictly speaking, this is not necessarily stiffness in the mathematical sense. A solver may need small time steps for accuracy rather than stability, or struggle with nonlinear convergence rather than stiffness itself.
Nevertheless, the practical result can be similar: smaller time steps, more solver iterations, and longer simulation times.

3. Difference in characteristic length: in chromatography we have the bulk and particle domain, the respective geometries differ in size by multiple orders of magnitude.
I guess this also boils down to time scales: things happen faster in small geometries.

4. Dispersion leads to bigger eigenvalues but also disperses the solution so it reduces strong gradients and the stiffness connected to them.

5. Column geometries can cause stiffness indirectly: interstitial velocity is faster at smaller cross sections.

X. is there anything else, maybe something unrelated to time schale?

<details>
<summary><b>Click to learn more about what specialized time integration approaches mitigate this and why its still usually not applied</b></summary>
Specialised time integration methods can exploit different time scales to improve simulation performance. For example, implicit-explicit (IMEX) methods treat stiff processes implicitly and non-stiff processes explicitly, while multirate methods can advance different processes using different time-step sizes.
Both approaches require additional knowledge of the model structure and careful handling of interactions between processes. Seems hard to be integrated in a non-specialized but general code like CADET. Though CADET's modular structure might pave the way for future developments in this direction.
IMEX methods have been successfully demonstrated for chromatography models in scientific publications. 
However, they have not established a general performance advantage over conventional implicit integrators such as adaptive BDF, which remains a standard choice in established chromatography simulators, and no open source IMEX simulator is available.
https://www.sciencedirect.com/science/article/pii/S0168927417301812?utm_source=chatgpt.com
https://www.sciencedirect.com/science/article/pii/S0098135417304362?utm_source=chatgpt.com
https://www.sciencedirect.com/science/chapter/bookseries/abs/pii/B9780444642417501373
<details>

todo: how to stop the details thing here for the key takeaway?
Key takeaway: A slow simulation is not necessarily caused by a large model. Even a relatively small model can be computationally expensive if it contains very fast processes, restrictive stability limits, or difficult nonlinear behavior.

plan for the next notebooks: one notebook for each potential cause of stiffness, where we set up a chromatography problem that exercises exactly that, followed by advise how to mitigate this and honesty about limitations, we cannot always have a fast simulation.